# 04 — Dashboard Data Preparation
## MedIQ — Hospital Patient Analytics & Readmission Predictor
**Goal:** Prepare all export CSVs for Power BI — patient overview, risk distribution, department analysis, model performance, and SHAP feature importance.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os

# Load all data
df       = pd.read_csv('../data/processed/mediq_clean.csv')
preds    = pd.read_csv('../exports/predictions.csv')
shap_imp = pd.read_csv('../exports/shap_importance.csv')
risk_str = pd.read_csv('../exports/risk_stratification.csv')
dept     = pd.read_csv('../exports/department_analysis.csv')

print(f"✅ All data loaded")
print(f"   Patients     : {len(df):,}")
print(f"   Predictions  : {len(preds):,}")
print(f"   SHAP features: {len(shap_imp)}")

✅ All data loaded
   Patients     : 101,745
   Predictions  : 20,349
   SHAP features: 23


In [2]:
# KPI metrics for Power BI cards
kpi = {
    'total_patients'        : len(df),
    'high_risk_patients'    : int(df['readmitted_binary'].sum()),
    'readmission_rate_pct'  : round(df['readmitted_binary'].mean()*100, 2),
    'avg_age'               : round(df['age_numeric'].mean(), 1),
    'avg_time_in_hospital'  : round(df['time_in_hospital'].mean(), 1),
    'avg_medications'       : round(df['num_medications'].mean(), 1),
    'avg_lab_procedures'    : round(df['num_lab_procedures'].mean(), 1),
    'model_auc'             : 0.61,
    'model_recall'          : 47.6,
    'optimal_threshold'     : 0.25
}

kpi_df = pd.DataFrame([kpi])
kpi_df.to_csv('../exports/kpi_metrics.csv', index=False)

print("✅ KPI metrics saved")
print(f"\n=== MEDIQ KPIs ===")
for k, v in kpi.items():
    print(f"   {k:<30} : {v}")

✅ KPI metrics saved

=== MEDIQ KPIs ===
   total_patients                 : 101745
   high_risk_patients             : 11352
   readmission_rate_pct           : 11.16
   avg_age                        : 66.0
   avg_time_in_hospital           : 4.4
   avg_medications                : 16.0
   avg_lab_procedures             : 43.1
   model_auc                      : 0.61
   model_recall                   : 47.6
   optimal_threshold              : 0.25


In [3]:
# Age group distribution
age_dist = df.groupby('age').agg(
    total_patients    = ('readmitted_binary', 'count'),
    high_risk         = ('readmitted_binary', 'sum'),
    readmission_rate  = ('readmitted_binary', 'mean'),
    avg_stay          = ('time_in_hospital', 'mean'),
    avg_medications   = ('num_medications', 'mean')
).reset_index()
age_dist['readmission_rate'] = (age_dist['readmission_rate']*100).round(2)
age_dist['avg_stay']         = age_dist['avg_stay'].round(2)
age_dist['avg_medications']  = age_dist['avg_medications'].round(2)

# Sort age groups correctly
age_order = ['[0-10)','[10-20)','[20-30)','[30-40)','[40-50)',
             '[50-60)','[60-70)','[70-80)','[80-90)','[90-100)']
age_dist['age_order'] = age_dist['age'].map(
    {a:i for i,a in enumerate(age_order)})
age_dist = age_dist.sort_values('age_order').drop('age_order', axis=1)

# Diagnosis distribution
diag_dist = df.groupby('diag_1_category').agg(
    total_patients   = ('readmitted_binary', 'count'),
    high_risk        = ('readmitted_binary', 'sum'),
    readmission_rate = ('readmitted_binary', 'mean'),
    avg_stay         = ('time_in_hospital', 'mean')
).reset_index()
diag_dist['readmission_rate'] = (diag_dist['readmission_rate']*100).round(2)
diag_dist['avg_stay']         = diag_dist['avg_stay'].round(2)
diag_dist = diag_dist.sort_values('readmission_rate', ascending=False)

age_dist.to_csv('../exports/age_distribution.csv', index=False)
diag_dist.to_csv('../exports/diagnosis_distribution.csv', index=False)

print("✅ Age and diagnosis exports saved")
print(f"\nAge groups: {len(age_dist)}")
print(f"Diagnoses : {len(diag_dist)}")
print(f"\nAge distribution:")
print(age_dist[['age','total_patients','readmission_rate']].to_string(index=False))

✅ Age and diagnosis exports saved

Age groups: 10
Diagnoses : 9

Age distribution:
     age  total_patients  readmission_rate
  [0-10)             161              1.86
 [10-20)             691              5.79
 [20-30)            1656             14.25
 [30-40)            3775             11.23
 [40-50)            9683             10.60
 [50-60)           17254              9.67
 [60-70)           22476             11.13
 [70-80)           26064             11.77
 [80-90)           17192             12.08
[90-100)            2793             11.10


In [4]:
# Risk distribution for Power BI
risk_dist = df.groupby('number_inpatient').agg(
    total_patients   = ('readmitted_binary', 'count'),
    high_risk        = ('readmitted_binary', 'sum'),
    readmission_rate = ('readmitted_binary', 'mean')
).reset_index()
risk_dist['readmission_rate'] = (risk_dist['readmission_rate']*100).round(2)
risk_dist = risk_dist[risk_dist['total_patients'] >= 50]

# Insulin analysis
insulin_dist = df.groupby('insulin').agg(
    total_patients   = ('readmitted_binary', 'count'),
    high_risk        = ('readmitted_binary', 'sum'),
    readmission_rate = ('readmitted_binary', 'mean'),
    avg_stay         = ('time_in_hospital', 'mean')
).reset_index()
insulin_dist['readmission_rate'] = (insulin_dist['readmission_rate']*100).round(2)
insulin_dist['avg_stay']         = insulin_dist['avg_stay'].round(2)
insulin_dist = insulin_dist.sort_values('readmission_rate', ascending=False)

# Time in hospital distribution
time_dist = df.groupby('time_in_hospital').agg(
    total_patients   = ('readmitted_binary', 'count'),
    readmission_rate = ('readmitted_binary', 'mean')
).reset_index()
time_dist['readmission_rate'] = (time_dist['readmission_rate']*100).round(2)

risk_dist.to_csv('../exports/inpatient_risk.csv', index=False)
insulin_dist.to_csv('../exports/insulin_analysis.csv', index=False)
time_dist.to_csv('../exports/time_distribution.csv', index=False)

print("✅ Risk and medication exports saved")
print(f"\nInsulin analysis:")
print(insulin_dist[['insulin','total_patients','readmission_rate']].to_string(index=False))

✅ Risk and medication exports saved

Insulin analysis:
insulin  total_patients  readmission_rate
   Down           12216             13.90
     Up           11315             12.99
 Steady           30843             11.12
     No           47371             10.03


In [5]:
# Risk probability distribution
prob_dist = pd.cut(
    preds['risk_probability'],
    bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    labels=['0-10%','10-20%','20-30%','30-40%','40-50%',
            '50-60%','60-70%','70-80%','80-90%','90-100%']
).value_counts().reset_index()
prob_dist.columns = ['risk_bucket', 'patient_count']
prob_dist = prob_dist.sort_values('risk_bucket')

# Actual vs predicted summary
perf_summary = pd.DataFrame({
    'metric'   : ['ROC-AUC','Recall (High Risk)','Precision (High Risk)',
                  'F1 Score','Accuracy','Optimal Threshold'],
    'value'    : [0.61, 47.6, 15.0, 23.1, 65.0, 0.25],
    'benchmark': [0.65, 50.0, 20.0, 25.0, 70.0, 0.50],
    'status'   : ['Near Benchmark','Near Benchmark','Below Benchmark',
                  'Near Benchmark','Below Benchmark','Optimised']
})

# SHAP top 10 for dashboard
shap_top10 = shap_imp.head(10)

prob_dist.to_csv('../exports/risk_probability_dist.csv', index=False)
perf_summary.to_csv('../exports/model_performance.csv', index=False)
shap_top10.to_csv('../exports/shap_top10.csv', index=False)

print("✅ Model performance exports saved")
print(f"\nModel performance summary:")
print(perf_summary.to_string(index=False))

✅ Model performance exports saved

Model performance summary:
               metric  value  benchmark          status
              ROC-AUC   0.61       0.65  Near Benchmark
   Recall (High Risk)  47.60      50.00  Near Benchmark
Precision (High Risk)  15.00      20.00 Below Benchmark
             F1 Score  23.10      25.00  Near Benchmark
             Accuracy  65.00      70.00 Below Benchmark
    Optimal Threshold   0.25       0.50       Optimised


In [6]:
# Final complete export
print("=" * 55)
print("   MEDIQ — DASHBOARD DATA PREPARATION COMPLETE")
print("=" * 55)

export_files = {
    'kpi_metrics.csv'           : 'KPI cards for Power BI',
    'age_distribution.csv'      : 'Age group analysis',
    'diagnosis_distribution.csv': 'Diagnosis risk breakdown',
    'inpatient_risk.csv'        : 'Prior visits risk curve',
    'insulin_analysis.csv'      : 'Insulin usage impact',
    'time_distribution.csv'     : 'Hospital stay analysis',
    'risk_stratification.csv'   : 'Risk tier breakdown',
    'department_analysis.csv'   : 'Department readmission rates',
    'medication_impact.csv'     : 'Medication change impact',
    'high_risk_segments.csv'    : 'Highest risk segments',
    'risk_probability_dist.csv' : 'Model risk distribution',
    'model_performance.csv'     : 'Model metrics vs benchmark',
    'shap_top10.csv'            : 'Top 10 SHAP features',
    'predictions.csv'           : 'Full patient predictions',
}

print(f"\n📁 exports/ folder — {len(export_files)} files ready for Power BI:")
total_rows = 0
for fname, desc in export_files.items():
    path = f'../exports/{fname}'
    if os.path.exists(path):
        rows = len(pd.read_csv(path))
        total_rows += rows
        print(f"   ✅ {fname:<40} {rows:>6} rows  — {desc}")
    else:
        print(f"   ❌ {fname} — MISSING")

print(f"\n   Total rows across all exports: {total_rows:,}")
print(f"\n📁 models/ folder:")
for f in os.listdir('../models'):
    print(f"   ✅ {f}")
print(f"\nNext → Flask Web App")
print("=" * 55)

   MEDIQ — DASHBOARD DATA PREPARATION COMPLETE

📁 exports/ folder — 14 files ready for Power BI:
   ✅ kpi_metrics.csv                               1 rows  — KPI cards for Power BI
   ✅ age_distribution.csv                         10 rows  — Age group analysis
   ✅ diagnosis_distribution.csv                    9 rows  — Diagnosis risk breakdown
   ✅ inpatient_risk.csv                           11 rows  — Prior visits risk curve
   ✅ insulin_analysis.csv                          4 rows  — Insulin usage impact
   ✅ time_distribution.csv                        14 rows  — Hospital stay analysis
   ✅ risk_stratification.csv                       4 rows  — Risk tier breakdown
   ✅ department_analysis.csv                      15 rows  — Department readmission rates
   ✅ medication_impact.csv                         4 rows  — Medication change impact
   ✅ high_risk_segments.csv                        9 rows  — Highest risk segments
   ✅ risk_probability_dist.csv                    10 rows  — M